# AlphaEarth Embeddings → Outlier Scores Pipeline

Scores WorldCereal samples using **64-dim AlphaEarth embeddings** fetched directly from [Source Cooperative COG tiles](https://source.coop/tge-labs/aef/v1/annual).  No GEE export needed — embeddings are sampled on-the-fly from the public S3 COGs.

## Part 1 — Fetch Embeddings from COGs

| # | Cell | Description |
|---|------|-------------|
| 0 | **GDAL env + index** | Set S3 credentials, load STAC geoparquet index, define constants (`AE_SCALE`, `AE_NODATA`) |
| 1 | *inspect hits* | `hits.head()` — spatial join result for two test points |
| 2 | *asset structure* | Show raw `assets` dict from one index row |
| 3 | **Smoke test** | Sample 5 manually chosen points and verify non-null embeddings |
| 4 | **Bulk fetch function** | `fetch_region_embeddings_from_cogs()` — parallel, year-aware, skip-existing |
| 5 | **Run for Southern Asia** | Fetch all 75k points → `alphaearth_cog_embeddings.duckdb` |
| 6 | **Verify DuckDB** | Row counts, year distribution, value stats |

## Part 2 — Outlier Scoring

| # | Section | Description |
|---|---------|-------------|
| 1 | **Parameters** | Paths & knobs — point `EMBEDDINGS_DB_PATH` at the COG DuckDB |
| 2 | **Inspect DuckDB** | Schema, deduplication |
| 2b | **Deduplicate** | Keep one row per `sample_id` |
| 2c | **Regional coverage** | Cross-check against global parquet |
| 3 | **Load region parquet** | Clean anomaly cols, check overlap |
| 4 | **Prepare embeddings** | Rename `A00`→`embedding_0`, join metadata |
| 5 | **Class mappings** | Local JSON or SharePoint |
| 6–7 | **LC10 / CTY24 scoring** | `run_pipeline(embeddings_df=...)` |
| 8–9 | **Merge & write back** | Save scored parquet |

> **COG encoding:** `int8`, nodata=`-128`, scale=`1/127` → float32 in `[-1, 1]`.  
> Band names `A00`–`A63` in the COG match the DuckDB column names exactly.


In [ ]:
# ==============================================================
# GDAL / rasterio environment — MUST be set before any COG access
# ==============================================================
import os
os.environ["AWS_NO_SIGN_REQUEST"]                = "YES"   # public Source Cooperative bucket
os.environ["AWS_DEFAULT_REGION"]                 = "us-west-2"
os.environ["GDAL_HTTP_MERGE_CONSECUTIVE_RANGES"] = "YES"
os.environ["GDAL_HTTP_MULTIPLEX"]                = "YES"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"]       = "EMPTY_DIR"
os.environ["CPL_VSIL_CURL_CACHE_SIZE"]           = "200000000"  # 200 MB tile cache

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from pyproj import Transformer
from shapely.geometry import Point

# ==============================================================
# Load STAC geoparquet index (local file)
# ==============================================================
INDEX_PATH = Path("/home/{path}/{fold}/TestFolder/wc_outliers/aef_index_stac_geoparquet.parquet")
assert INDEX_PATH.exists(), f"Index not found: {INDEX_PATH}"

index = gpd.read_parquet(INDEX_PATH)
print(f"Index loaded : {len(index):,} rows  |  CRS: {index.crs}")

# Pre-process once: extract year integer + convert S3 URL → /vsis3/ path
index["year_int"] = index["datetime"].dt.year
index["cog_url"]  = index["assets"].apply(
    # s3://us-west-2.opendata.source.coop/…  →  /vsis3/us-west-2.opendata.source.coop/…
    lambda x: "/vsis3/" + x["data"]["href"][5:]
)

print(f"Years in index : {sorted(index['year_int'].unique())}")
print(f"Total COG tiles: {index['cog_url'].nunique():,}")
print(f"Sample COG URL : {index['cog_url'].iloc[1000]}")

# ==============================================================
# COG encoding constants  (verified via rasterio + GDAL tags)
# ==============================================================
# dtype : int8   (nodata = -128, range = -127 … 127)
# scale : 1/127  → divide raw int8 by 127 to get float32 in [-1, 1]
# Band names: A00 … A63 (matches DuckDB column names)
AE_NODATA  = -128
AE_SCALE   = 1.0 / 127.0
AE_N_DIMS  = 64
AE_BAND_NAMES = [f"A{i:02d}" for i in range(AE_N_DIMS)]
print(f"Band names: {AE_BAND_NAMES[:4]} … {AE_BAND_NAMES[-4:]}")


In [ ]:
# # ==============================================================
# # SMOKE TEST — verify COG sampling for a handful of S Asia points
# # ==============================================================
# # Each row: (sample_id, lat, lon, year)
# smoke_pts = pd.DataFrame({
#     "sample_id": ["smoke_01", "smoke_02", "smoke_03", "smoke_04", "smoke_05"],
#     "lat": [27.50,  28.10,  23.50,  12.50,  30.20],
#     "lon": [80.20,  79.80,  72.80,  77.50,  75.10],
#     "year": [2024,  2024,   2018,   2019,   2022],
# })

# def sample_cog_for_points(pts_df, index_gdf):
#     """
#     Given a DataFrame with [sample_id, lat, lon, year], spatial-join
#     to the STAC index (per year), then sample each matched COG.

#     Returns a DataFrame with sample_id + A00…A63 float values.
#     Rows where the point falls on nodata are kept with NaN embeddings.
#     """
#     gdf = gpd.GeoDataFrame(
#         pts_df,
#         geometry=[Point(lon, lat) for lat, lon in zip(pts_df["lat"], pts_df["lon"])],
#         crs=index_gdf.crs,   # match index CRS (OGC:CRS84 ≡ EPSG:4326)
#     )

#     # Cache Transformer objects to avoid repeated creation
#     _tx_cache: dict[int, Transformer] = {}

#     records = []
#     for year, grp in gdf.groupby("year"):
#         idx_year = index_gdf[index_gdf["year_int"] == year][
#             ["cog_url", "proj:epsg", "geometry"]
#         ]
#         if idx_year.empty:
#             print(f"  [year={year}] No index tiles found — skipping {len(grp)} points")
#             continue

#         hits = gpd.sjoin(grp, idx_year, how="left", predicate="within")

#         # Points not within any tile (coast/border edge): try intersects fallback
#         missed = hits[hits["cog_url"].isna()]
#         if not missed.empty:
#             missed_geom = missed.drop(columns=["index_right", "cog_url", "proj:epsg"],
#                                       errors="ignore")
#             hits_fallback = gpd.sjoin(
#                 missed_geom, idx_year, how="left", predicate="intersects"
#             )
#             hits = pd.concat(
#                 [hits[hits["cog_url"].notna()], hits_fallback], ignore_index=True
#             )

#         for cog_url, tile_grp in hits[hits["cog_url"].notna()].groupby("cog_url"):
#             epsg_str = tile_grp["proj:epsg"].iloc[0]
#             epsg = int(epsg_str.split(":")[-1])

#             if epsg not in _tx_cache:
#                 _tx_cache[epsg] = Transformer.from_crs(
#                     "EPSG:4326", epsg, always_xy=True
#                 )
#             tx = _tx_cache[epsg]

#             xy = [tx.transform(r.lon, r.lat) for _, r in tile_grp.iterrows()]

#             with rasterio.open(cog_url) as src:
#                 raw = np.array(list(src.sample(xy)), dtype=np.int8)  # (n_pts, 64)

#             for i, (_, row) in enumerate(tile_grp.iterrows()):
#                 rv = raw[i]
#                 is_nodata = np.all(rv == AE_NODATA)
#                 emb = (None,) * AE_N_DIMS if is_nodata else tuple(
#                     (rv.astype(np.float32) * AE_SCALE).tolist()
#                 )
#                 rec = {
#                     "sample_id": row["sample_id"],
#                     "lat": row["lat"], "lon": row["lon"], "year": year,
#                     "cog_url": cog_url, "valid": not is_nodata,
#                 }
#                 for j, name in enumerate(AE_BAND_NAMES):
#                     rec[name] = emb[j]
#                 records.append(rec)

#         # Unmatched even after fallback
#         unmatched = hits[hits["cog_url"].isna()]
#         for _, row in unmatched.iterrows():
#             rec = {"sample_id": row["sample_id"], "lat": row["lat"],
#                    "lon": row["lon"], "year": year, "cog_url": None, "valid": False}
#             for name in AE_BAND_NAMES:
#                 rec[name] = None
#             records.append(rec)

#     return pd.DataFrame(records)


# df_smoke = sample_cog_for_points(smoke_pts, index)
# display(df_smoke[["sample_id", "lat", "lon", "year", "valid", "cog_url"] +
#                   AE_BAND_NAMES[:4]])
# print(f"\n{df_smoke['valid'].sum()}/{len(df_smoke)} points have valid embeddings")


In [ ]:
# ==============================================================
# BULK FETCH — scalable COG sampler for any region parquet
# ==============================================================
import concurrent.futures
import duckdb
from tqdm.auto import tqdm

def _fetch_one_tile(args):
    """Worker: open one COG, sample all points in it, return list of dicts."""
    cog_url, tile_grp, epsg = args
    tx = Transformer.from_crs("EPSG:4326", epsg, always_xy=True)
    rows_out = []
    try:
        xy = [tx.transform(r.lon, r.lat) for _, r in tile_grp.iterrows()]
        with rasterio.open(cog_url) as src:
            raw = np.array(list(src.sample(xy)), dtype=np.int8)
        for i, (_, row) in enumerate(tile_grp.iterrows()):
            rv = raw[i]
            is_nodata = np.all(rv == AE_NODATA)
            emb = (rv.astype(np.float32) * AE_SCALE).tolist() if not is_nodata else [None] * AE_N_DIMS
            rec = {"sample_id": row["sample_id"], "lat": float(row["lat"]),
                   "lon": float(row["lon"]), "year": int(row["year"]),
                   "valid": not is_nodata}
            for j, name in enumerate(AE_BAND_NAMES):
                rec[name] = emb[j]
            rows_out.append(rec)
    except Exception as e:
        # On error mark all points in this tile as invalid
        for _, row in tile_grp.iterrows():
            rec = {"sample_id": row["sample_id"], "lat": float(row["lat"]),
                   "lon": float(row["lon"]), "year": int(row["year"]),
                   "valid": False, "_error": str(e)}
            for name in AE_BAND_NAMES:
                rec[name] = None
            rows_out.append(rec)
    return rows_out


def fetch_region_embeddings_from_cogs(
    region_df,        # DataFrame: sample_id, lat, lon, year
    index_gdf,        # Pre-processed STAC index (with year_int, cog_url columns)
    output_db_path,   # Path to output DuckDB (will be created / appended)
    n_workers=6,      # Parallel COG readers (threads)
    skip_existing=True,  # Skip sample_ids already in output_db
    write_batch=2000, # Rows buffered before committing to DuckDB
):
    """
    Fetch 64-dim AlphaEarth embeddings for all points in `region_df`
    by sampling COG tiles from Source Cooperative.

    Output DuckDB table: `embeddings`  with columns
        sample_id, lat, lon, year, valid, A00..A63
    """
    output_db_path = Path(output_db_path)
    output_db_path.parent.mkdir(parents=True, exist_ok=True)

    # ── Init output DuckDB ────────────────────────────────────────────
    con = duckdb.connect(str(output_db_path))
    col_defs = ", ".join([f'"{n}" FLOAT' for n in AE_BAND_NAMES])
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS embeddings (
            sample_id VARCHAR,
            lat DOUBLE, lon DOUBLE, year INTEGER,
            valid BOOLEAN,
            {col_defs}
        )
    """)
    con.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_sid ON embeddings (sample_id)")

    # ── Determine which sample_ids to fetch ──────────────────────────
    if skip_existing:
        existing_ids = set(
            con.execute("SELECT sample_id FROM embeddings").fetchdf()["sample_id"].tolist()
        )
        todo_df = region_df[~region_df["sample_id"].isin(existing_ids)].copy()
        print(f"Already in DB     : {len(existing_ids):,}")
        print(f"To fetch          : {len(todo_df):,} / {len(region_df):,}")
    else:
        todo_df = region_df.copy()
        print(f"Fetching (no skip): {len(todo_df):,} points")
    con.close()

    if todo_df.empty:
        print("Nothing to fetch — all sample_ids already present.")
        return

    # ── Build task list: one task per (year, cog_url) ────────────────
    gdf = gpd.GeoDataFrame(
        todo_df,
        geometry=[Point(lon, lat) for lon, lat in zip(todo_df["lon"], todo_df["lat"])],
        crs=index_gdf.crs,   # match index CRS (OGC:CRS84 ≡ EPSG:4326)
    )

    tasks = []          # list of (cog_url, tile_grp, epsg)
    unmatched_count = 0

    for year, grp in tqdm(gdf.groupby("year"), desc="Spatial-join years", unit="yr"):
        print(f"  [year={year}] {len(grp):,} points")
        idx_year = index_gdf[index_gdf["year_int"] == year][
            ["cog_url", "proj:epsg", "geometry"]
        ]
        if idx_year.empty:
            unmatched_count += len(grp)
            continue

        hits = gpd.sjoin(grp, idx_year, how="left", predicate="within")

        # Fallback for boundary points
        missed = hits[hits["cog_url"].isna()]
        if not missed.empty:
            m2 = missed.drop(columns=["index_right","cog_url","proj:epsg"], errors="ignore")
            hits2 = gpd.sjoin(m2, idx_year, how="left", predicate="intersects")
            hits = pd.concat([hits[hits["cog_url"].notna()], hits2], ignore_index=True)

        truly_unmatched = hits[hits["cog_url"].isna()]
        unmatched_count += len(truly_unmatched)

        for cog_url, tile_grp in hits[hits["cog_url"].notna()].groupby("cog_url"):
            print(f"    [cog_url={cog_url}] {len(tile_grp):,} points")
            epsg = int(tile_grp["proj:epsg"].iloc[0].split(":")[-1])
            tasks.append((cog_url, tile_grp[["sample_id","lat","lon","year"]], epsg))

    print(f"Tile tasks        : {len(tasks):,}")
    print(f"Unmatched points  : {unmatched_count:,}  (no COG tile found)")

    # ── Fetch tiles in parallel and write to DuckDB ───────────────────
    buffer   = []
    n_valid  = 0
    n_nodata = 0
    n_errors = 0

    def flush_buffer(buf, con):
        if not buf:
            return
        df_buf = pd.DataFrame(buf)
        # Upsert: insert or ignore for already-existing sample_ids
        con.register("_buf", df_buf)
        con.execute("INSERT OR IGNORE INTO embeddings SELECT * FROM _buf")
        con.unregister("_buf")

    pbar = tqdm(total=len(tasks), desc="Fetching COG tiles", unit="tile")
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as exe:
        futures = {exe.submit(_fetch_one_tile, t): t for t in tasks}
        for fut in concurrent.futures.as_completed(futures):
            rows = fut.result()
            buffer.extend(rows)
            for r in rows:
                if r.get("_error"):
                    n_errors += 1
                elif r["valid"]:
                    n_valid += 1
                else:
                    n_nodata += 1

            if len(buffer) >= write_batch:
                con2 = duckdb.connect(str(output_db_path))
                flush_buffer(buffer, con2)
                con2.close()
                buffer.clear()
            pbar.update(1)

    pbar.close()

    # Final flush
    if buffer:
        con2 = duckdb.connect(str(output_db_path))
        flush_buffer(buffer, con2)
        con2.close()

    # ── Summary ───────────────────────────────────────────────────────
    con = duckdb.connect(str(output_db_path), read_only=True)
    total_in_db = con.execute("SELECT COUNT(*) FROM embeddings").fetchone()[0]
    valid_in_db = con.execute("SELECT COUNT(*) FROM embeddings WHERE valid=TRUE").fetchone()[0]
    con.close()

    print(f"\n✓ Done  —  {output_db_path.name}")
    print(f"  Valid embeddings : {n_valid:,}")
    print(f"  Nodata points    : {n_nodata:,}  (no coverage at pixel)")
    print(f"  Errors           : {n_errors:,}")
    print(f"  Total in DB now  : {total_in_db:,}  (valid: {valid_in_db:,})")

print("fetch_region_embeddings_from_cogs() defined and ready.")


In [ ]:

# ==============================================================
# MINI TEST — fetch ~20 rows → write parquet → read back & verify
# Prerequisites: run cell 2 (GDAL env + index + constants)
#                run cell 6 (defines _fetch_one_tile)
# ==============================================================
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point

REGION_PARQUET = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier/Eastern_Asia.parquet")
# TEST_PARQUET   = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier/alphaearth_cog_test.parquet")
# TEST_N_ROWS    = 20   # stop after accumulating at least this many point rows

# # ── Load points ───────────────────────────────────────────────────────
# pts = pd.read_parquet(str(REGION_PARQUET), columns=["sample_id", "lat", "lon", "year"])
# pts = pts.drop_duplicates(subset="sample_id").reset_index(drop=True)
# print(f"Total points available: {len(pts):,}")

# # ── Build just enough tasks to cover TEST_N_ROWS points ──────────────
# gdf = gpd.GeoDataFrame(
#     pts,
#     geometry=[Point(lon, lat) for lon, lat in zip(pts["lon"], pts["lat"])],
#     crs=index.crs,
# )

# test_tasks = []
# accumulated = 0
# for year, grp in gdf.groupby("year"):
#     if accumulated >= TEST_N_ROWS:
#         break
#     idx_year = index[index["year_int"] == year][["cog_url", "proj:epsg", "geometry"]]
#     if idx_year.empty:
#         continue
#     hits = gpd.sjoin(grp, idx_year, how="left", predicate="within")
#     for cog_url, tile_grp in hits[hits["cog_url"].notna()].groupby("cog_url"):
#         if accumulated >= TEST_N_ROWS:
#             break
#         epsg = int(tile_grp["proj:epsg"].iloc[0].split(":")[-1])
#         test_tasks.append((cog_url, tile_grp[["sample_id", "lat", "lon", "year"]], epsg))
#         accumulated += len(tile_grp)

# print(f"Tasks to run   : {len(test_tasks)}")
# print(f"Points covered : ~{accumulated}")

# # ── Fetch (serial — easy to debug) ───────────────────────────────────
# all_rows = []
# for cog_url, tile_grp, epsg in test_tasks:
#     rows = _fetch_one_tile((cog_url, tile_grp, epsg))
#     n_valid = sum(r["valid"] for r in rows)
#     print(f"  {cog_url.split('/')[-1]}  → {len(rows)} pts,  {n_valid} valid")
#     all_rows.extend(rows)

# df_test = pd.DataFrame(all_rows)
# df_test.drop(columns=["_error"], errors="ignore", inplace=True)  # only present on exceptions

# # ── Write parquet ─────────────────────────────────────────────────────
# df_test.to_parquet(str(TEST_PARQUET), index=False)
# print(f"\n✓ Written {len(df_test)} rows  →  {TEST_PARQUET.name}")

# # ── Read back & verify ────────────────────────────────────────────────
# df_v = pd.read_parquet(str(TEST_PARQUET))
# print(f"\nRead-back verification:")
# print(f"  Rows        : {len(df_v)}")
# print(f"  Valid       : {df_v['valid'].sum()} / {len(df_v)}")
# print(f"  A00 range   : [{df_v['A00'].min():.4f},  {df_v['A00'].max():.4f}]   (expected [-1, 1])")
# print(f"  lat range   : [{df_v['lat'].min():.3f},  {df_v['lat'].max():.3f}]    (expected ~5–40°N)")
# print(f"  lon range   : [{df_v['lon'].min():.3f},  {df_v['lon'].max():.3f}]    (expected ~60–100°E)")
# display(df_v[["sample_id", "lat", "lon", "year", "valid", "A00", "A01", "A02"]].head(10))


In [ ]:

# ==============================================================
# FULL FETCH → resumable parquet chunks (2 000 rows each)
# ==============================================================
# Output layout:
#   CHUNKS_DIR/chunk_000000.parquet   ← up to WRITE_BATCH rows each
#   CHUNKS_DIR/chunk_000001.parquet
#   ...
#   FINAL_PARQUET                     ← merged at the end
#
# Restart-safe:  existing chunks are scanned on startup; their
# sample_ids are skipped so only missing points are fetched.
# ==============================================================
import concurrent.futures
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.geometry import Point
from tqdm.auto import tqdm

REGION_PARQUET = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier/Eastern_Africa.parquet")
CHUNKS_DIR     = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier/alphaearth_cog_chunks_Eastern_Africa")
FINAL_PARQUET  = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier/alphaearth_cog_embeddings_Eastern_Africa.parquet")
WRITE_BATCH    = 1_000   # rows per chunk file
N_WORKERS      = 16

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load all points ───────────────────────────────────────────────────
pts = pd.read_parquet(str(REGION_PARQUET), columns=["sample_id", "lat", "lon", "year"])
pts = pts.drop_duplicates(subset="sample_id").reset_index(drop=True)
print(f"Total unique points : {len(pts):,}")

# ── Resume: collect sample_ids already saved in existing chunks ───────
existing_chunks = sorted(CHUNKS_DIR.glob("chunk_*.parquet"))
if existing_chunks:
    existing_ids = set(
        pd.concat(
            [pd.read_parquet(str(f), columns=["sample_id"]) for f in existing_chunks]
        )["sample_id"].dropna().tolist()
    )
    print(f"Existing chunks     : {len(existing_chunks)}  ({len(existing_ids):,} sample_ids already saved)")
else:
    existing_ids = set()
    print("No existing chunks — starting fresh.")

chunk_idx = len(existing_chunks)   # next chunk number

# ── Filter to remaining points ────────────────────────────────────────
todo_pts = pts[~pts["sample_id"].isin(existing_ids)].copy().reset_index(drop=True)
print(f"Remaining to fetch  : {len(todo_pts):,} / {len(pts):,}")

if todo_pts.empty:
    print("Nothing to fetch — all sample_ids already present.")
else:
    # ── Build full task list ──────────────────────────────────────────
    gdf = gpd.GeoDataFrame(
        todo_pts,
        geometry=[Point(lon, lat) for lon, lat in zip(todo_pts["lon"], todo_pts["lat"])],
        crs=index.crs,
    )
    tasks, unmatched = [], 0
    for year, grp in tqdm(gdf.groupby("year"), desc="Building task list", unit="yr"):
        idx_year = index[index["year_int"] == year][["cog_url", "proj:epsg", "geometry"]]
        if idx_year.empty:
            unmatched += len(grp)
            continue
        hits = gpd.sjoin(grp, idx_year, how="left", predicate="within")
        missed = hits[hits["cog_url"].isna()]
        if not missed.empty:
            m2 = missed.drop(columns=["index_right", "cog_url", "proj:epsg"], errors="ignore")
            hits2 = gpd.sjoin(m2, idx_year, how="left", predicate="intersects")
            hits = pd.concat([hits[hits["cog_url"].notna()], hits2], ignore_index=True)
        unmatched += int(hits["cog_url"].isna().sum())
        for cog_url, tile_grp in hits[hits["cog_url"].notna()].groupby("cog_url"):
            epsg = int(tile_grp["proj:epsg"].iloc[0].split(":")[-1])
            tasks.append((cog_url, tile_grp[["sample_id", "lat", "lon", "year"]], epsg))

    print(f"Tile tasks   : {len(tasks):,}")
    print(f"Unmatched pts: {unmatched:,}")

    # ── Helper: write one chunk ───────────────────────────────────────
    def flush_chunk(buf, idx):
        df_c = pd.DataFrame(buf)
        df_c.drop(columns=["_error"], errors="ignore", inplace=True)
        path = CHUNKS_DIR / f"chunk_{idx:06d}.parquet"
        df_c.to_parquet(str(path), index=False)
        return path

    # ── Fetch tiles in parallel, flush every WRITE_BATCH rows ────────
    buffer = []
    n_valid = n_nodata = n_errors = 0

    pbar = tqdm(total=len(tasks), desc="Fetching COG tiles", unit="tile")
    with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as exe:
        futures = {exe.submit(_fetch_one_tile, t): t for t in tasks}
        for fut in concurrent.futures.as_completed(futures):
            rows = fut.result()
            buffer.extend(rows)
            for r in rows:
                if r.get("_error"):  n_errors += 1
                elif r["valid"]:     n_valid  += 1
                else:                n_nodata += 1
            if len(buffer) >= WRITE_BATCH:
                p = flush_chunk(buffer, chunk_idx)
                tqdm.write(f"  ✓ chunk_{chunk_idx:06d}.parquet  ({len(buffer)} rows)")
                chunk_idx += 1
                buffer.clear()
            pbar.update(1)
    pbar.close()

    if buffer:                          # flush remainder
        p = flush_chunk(buffer, chunk_idx)
        print(f"  ✓ chunk_{chunk_idx:06d}.parquet  ({len(buffer)} rows)  [final]")

    print(f"\nFetch done — valid: {n_valid:,}  nodata: {n_nodata:,}  errors: {n_errors:,}")

# ── Merge all chunks → single final parquet ───────────────────────────
all_chunks = sorted(CHUNKS_DIR.glob("chunk_*.parquet"))
print(f"\nMerging {len(all_chunks)} chunks …")
df_all = pd.concat([pd.read_parquet(str(f)) for f in all_chunks], ignore_index=True)
df_all.drop_duplicates(subset="sample_id", inplace=True)
df_all.to_parquet(str(FINAL_PARQUET), index=False)
print(f"✓ Final parquet : {len(df_all):,} rows  ({df_all['valid'].sum():,} valid)")
print(f"  Path : {FINAL_PARQUET}")
print(f"  A00 range : [{df_all['A00'].min():.4f},  {df_all['A00'].max():.4f}]")
display(df_all[["sample_id", "lat", "lon", "year", "valid", "A00", "A01"]].head(5))


In [ ]:

# ==============================================================
# MERGE ALL CHUNKS → final parquet
# Run after BOTH cell 8 (main fetch) and cell 10 (heavy tiles) finish.
# ==============================================================
import pandas as pd
from pathlib import Path

_DATA         = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier")
CHUNKS_DIR    = _DATA / "alphaearth_cog_chunks"
HEAVY_DIR     = CHUNKS_DIR / "heavy"
FINAL_PARQUET = _DATA / "alphaearth_cog_embeddings.parquet"

regular_chunks = sorted(CHUNKS_DIR.glob("chunk_*.parquet"))
heavy_chunks   = sorted(HEAVY_DIR.glob("chunk_h_*.parquet")) if HEAVY_DIR.exists() else []
all_chunks     = regular_chunks + heavy_chunks

print(f"Regular chunks : {len(regular_chunks)}")
print(f"Heavy chunks   : {len(heavy_chunks)}")
print(f"Total          : {len(all_chunks)}  →  merging …")
assert all_chunks, "No chunk files found — fetch hasn't started yet!"

merged_df = pd.concat([pd.read_parquet(str(f)) for f in all_chunks], ignore_index=True)
merged_df.drop_duplicates(subset="sample_id", inplace=True)

embedding_cols = [f"embedding_{i}" for i in range(64)]
merged_df.rename(columns={f"A{i:02d}": embedding_cols[i] for i in range(64)}, inplace=True)

merged_df.to_parquet(str(FINAL_PARQUET), index=False)
print(f"\n✓ Final parquet  : {len(merged_df):,} rows  ({merged_df['valid'].sum():,} valid)")
print(f"  Path            : {FINAL_PARQUET}")
print(f"  embedding_0 range: [{merged_df['embedding_0'].min():.4f},  {merged_df['embedding_0'].max():.4f}]")
display(merged_df[["sample_id","lat","lon","year","valid","embedding_0","embedding_1"]].head(5))


In [ ]:

# ==============================================================
# HEAVY-TILE SAMPLER  (cell 10)
#
# For tiles with >= HEAVY_THRESHOLD unsampled points, the COG is
# downloaded locally one-at-a-time then sampled from disk.
# Local disk sampling is ~100× faster than per-point HTTP reads.
# Tiles below the threshold are handled with normal COG sampling.
#
# Output:  alphaearth_cog_chunks/heavy/chunk_h_*.parquet
# No conflict with cell 8 writing to alphaearth_cog_chunks/ directly
# → safe to run in a separate kernel while cell 8 is still running.
# ==============================================================
import os, concurrent.futures
import geopandas as gpd, numpy as np, pandas as pd, rasterio
from pathlib import Path
from pyproj import Transformer
from rasterio.shutil import copy as _rio_copy
from shapely.geometry import Point
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────
_DATA           = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier")
REGION_PARQUET  = _DATA / "Eastern_Africa.parquet"
CHUNKS_DIR      = _DATA / "alphaearth_cog_chunks_eastern_africa"
HEAVY_DIR       = CHUNKS_DIR / "heavy"
TMP_DIR         = _DATA / "ae_heavy_tiles"

HEAVY_THRESHOLD = 1_000    # pts/tile above this → download locally first
WRITE_BATCH     = 1_000  # rows per output chunk file
LIGHT_WORKERS   = 8      # parallel threads for light-tile COG sampling

HEAVY_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# ── COG encoding constants ─────────────────────────────────────────────
AE_NODATA = -128;  AE_SCALE = 1.0 / 127.0
AE_N_DIMS = 64;    AE_BAND_NAMES = [f"A{i:02d}" for i in range(64)]
os.environ.update({"AWS_NO_SIGN_REQUEST": "YES", "AWS_DEFAULT_REGION": "us-west-2",
                   "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES",
                   "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
                   "CPL_VSIL_CURL_CACHE_SIZE": "200000000"})

# ── Index (reuse if already loaded in this kernel, else reload) ────────
try:
    len(index)
except NameError:
    _ip = Path("/home/{path}/{fold}/TestFolder/wc_outliers/aef_index_stac_geoparquet.parquet")
    index = gpd.read_parquet(_ip)
    index["year_int"] = index["datetime"].dt.year
    index["cog_url"]  = index["assets"].apply(lambda x: "/vsis3/" + x["data"]["href"][5:])
    print(f"Index loaded: {len(index):,} rows")

# ── Points & already-saved ids ─────────────────────────────────────────
pts = pd.read_parquet(str(REGION_PARQUET), columns=["sample_id","lat","lon","year"])
pts = pts.drop_duplicates("sample_id").reset_index(drop=True)

_done_chunks = (sorted(CHUNKS_DIR.glob("chunk_*.parquet")) +
                sorted(HEAVY_DIR.glob("chunk_h_*.parquet")))
_done_ids = set(
    pd.concat([pd.read_parquet(str(f), columns=["sample_id"])
               for f in _done_chunks])["sample_id"].dropna()
) if _done_chunks else set()

todo = pts[~pts["sample_id"].isin(_done_ids)].copy().reset_index(drop=True)
print(f"Total: {len(pts):,}  |  done: {len(_done_ids):,}  |  remaining: {len(todo):,}")
print(f"  (regular chunks: {len(sorted(CHUNKS_DIR.glob('chunk_*.parquet')))}"
      f"  heavy chunks: {len(sorted(HEAVY_DIR.glob('chunk_h_*.parquet')))})")

if not todo.empty:
    # ── Spatial join → split into heavy / light tasks ─────────────────
    gdf = gpd.GeoDataFrame(
        todo,
        geometry=[Point(lon, lat) for lon, lat in zip(todo["lon"], todo["lat"])],
        crs=index.crs,
    )
    heavy_tasks, light_tasks, unmatched = [], [], 0
    for year, grp in tqdm(gdf.groupby("year"), desc="Spatial join", unit="yr"):
        iy = index[index["year_int"] == year][["cog_url","proj:epsg","geometry"]]
        if iy.empty:
            unmatched += len(grp); continue
        h = gpd.sjoin(grp, iy, how="left", predicate="within")
        miss = h[h["cog_url"].isna()]
        if not miss.empty:
            m2 = miss.drop(columns=["index_right","cog_url","proj:epsg"], errors="ignore")
            h  = pd.concat([h[h["cog_url"].notna()],
                            gpd.sjoin(m2, iy, how="left", predicate="intersects")],
                           ignore_index=True)
        unmatched += int(h["cog_url"].isna().sum())
        for url, tg in h[h["cog_url"].notna()].groupby("cog_url"):
            ep = int(tg["proj:epsg"].iloc[0].split(":")[-1])
            (heavy_tasks if len(tg) >= HEAVY_THRESHOLD else light_tasks).append(
                (url, tg[["sample_id","lat","lon","year"]].copy(), ep))

    print(f"\nHeavy ≥{HEAVY_THRESHOLD} pts : {len(heavy_tasks)} tiles  "
          f"({sum(len(t[1]) for t in heavy_tasks):,} pts)  ← local download")
    print(f"Light  <{HEAVY_THRESHOLD} pts : {len(light_tasks)} tiles  "
          f"({sum(len(t[1]) for t in light_tasks):,} pts)  ← parallel COG")
    print(f"Unmatched             : {unmatched:,}")

    # ── Record builders ────────────────────────────────────────────────
    def _rec(row, rv):
        nd  = np.all(rv == AE_NODATA)
        emb = (rv.astype(np.float32) * AE_SCALE).tolist() if not nd else [None]*AE_N_DIMS
        return {"sample_id": row["sample_id"], "lat": float(row["lat"]),
                "lon": float(row["lon"]), "year": int(row["year"]), "valid": not nd,
                **dict(zip(AE_BAND_NAMES, emb))}

    def _err(row, e):
        return {"sample_id": row["sample_id"], "lat": float(row["lat"]),
                "lon": float(row["lon"]), "year": int(row["year"]),
                "valid": False, "_error": str(e), **{n: None for n in AE_BAND_NAMES}}

    # ── Heavy worker: download COG → sample from disk → delete ────────
    def _fetch_local(url, tg, ep):
        tx  = Transformer.from_crs("EPSG:4326", ep, always_xy=True)
        loc = TMP_DIR / url.split("/")[-1]
        out = []
        try:
            if not loc.exists():
                # Downloads the full tile locally (compressed GTiff ~50-400 MB).
                # rasterio streams the COG block-by-block; with merged HTTP ranges
                # this is typically 1-2 large requests vs. N per-point requests.
                _rio_copy(url, str(loc), driver="GTiff", compress="deflate")
            tqdm.write(f"  ↓ {loc.name}  {loc.stat().st_size/1e6:.0f} MB  ({len(tg)} pts)")
            xy = [tx.transform(r.lon, r.lat) for _, r in tg.iterrows()]
            with rasterio.open(str(loc)) as src:
                raw = np.array(list(src.sample(xy)), dtype=np.int8)
            for i, (_, r) in enumerate(tg.iterrows()):
                out.append(_rec(r, raw[i]))
        except Exception as e:
            tqdm.write(f"  ✗ {loc.name}: {e}")
            for _, r in tg.iterrows():
                out.append(_err(r, e))
        finally:
            if loc.exists():
                loc.unlink()   # always delete — don't accumulate disk usage
        return out

    # ── Light worker: normal per-point COG sampling ────────────────────
    def _fetch_cog(url, tg, ep):
        tx = Transformer.from_crs("EPSG:4326", ep, always_xy=True)
        out = []
        try:
            xy = [tx.transform(r.lon, r.lat) for _, r in tg.iterrows()]
            with rasterio.open(url) as src:
                raw = np.array(list(src.sample(xy)), dtype=np.int8)
            for i, (_, r) in enumerate(tg.iterrows()):
                out.append(_rec(r, raw[i]))
        except Exception as e:
            for _, r in tg.iterrows():
                out.append(_err(r, e))
        return out

    # ── Chunk writer state ─────────────────────────────────────────────
    _st = {"nv": 0, "nn": 0, "ne": 0,
           "idx": len(sorted(HEAVY_DIR.glob("chunk_h_*.parquet"))),
           "buf": []}

    def _push(rows):
        _st["buf"].extend(rows)
        for r in rows:
            if r.get("_error"):  _st["ne"] += 1
            elif r["valid"]:     _st["nv"] += 1
            else:                _st["nn"] += 1
        if len(_st["buf"]) >= WRITE_BATCH:
            df = pd.DataFrame(_st["buf"])
            df.drop(columns=["_error"], errors="ignore", inplace=True)
            p  = HEAVY_DIR / f"chunk_h_{_st['idx']:06d}.parquet"
            df.to_parquet(str(p), index=False)
            tqdm.write(f"  ✓ {p.name}  ({len(df)} rows)")
            _st["idx"] += 1; _st["buf"].clear()

    # ── Run heavy tiles (serial — one download at a time = one temp file) ──
    print(f"\n── Heavy tiles: local download (serial) ──────────────────────")
    for url, tg, ep in tqdm(heavy_tasks, desc="Heavy tiles", unit="tile"):
        _push(_fetch_local(url, tg, ep))

    # ── Run light tiles (parallel COG sampling) ────────────────────────
    if light_tasks:
        print(f"\n── Light tiles: COG sampling ({LIGHT_WORKERS} threads) ───────────────")
        with concurrent.futures.ThreadPoolExecutor(max_workers=LIGHT_WORKERS) as exe:
            futs = {exe.submit(_fetch_cog, *t): t for t in light_tasks}
            for f in tqdm(concurrent.futures.as_completed(futs),
                          total=len(light_tasks), desc="Light tiles", unit="tile"):
                _push(f.result())

    # ── Final flush ────────────────────────────────────────────────────
    if _st["buf"]:
        df = pd.DataFrame(_st["buf"])
        df.drop(columns=["_error"], errors="ignore", inplace=True)
        p  = HEAVY_DIR / f"chunk_h_{_st['idx']:06d}.parquet"
        df.to_parquet(str(p), index=False)
        print(f"  ✓ {p.name}  ({len(df)} rows)  [final flush]")

    print(f"\n✓ Done — valid: {_st['nv']:,}  nodata: {_st['nn']:,}  errors: {_st['ne']:,}")
    print(f"  Output dir: {HEAVY_DIR}")
else:
    print("Nothing to fetch — all points already saved.")

print("\n→ Run cell 9 (merge) after both this cell and cell 8 are complete.")


---
## Part 2 — Outlier Scoring on AlphaEarth Embeddings

The cells below use the DuckDB produced in Part 1 (`alphaearth_cog_embeddings.duckdb`) to run the `run_pipeline` outlier detection — the same pipeline as the standard Presto-based workflow, but using AlphaEarth 64-dim embeddings fetched directly from Source Cooperative COGs.

> **If you already have the COG DuckDB** from a previous run, you can skip Part 1 and jump straight here. Just set `COG_EMBEDDINGS_DB` in the parameters cell below to point to it.

## 1) Parameters

Edit the paths and knobs below before running any other cell.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from loguru import logger
from tqdm.auto import tqdm

from outlier_embeddings.anomaly import run_pipeline

# ============================================================
# REGION TAG  — change this to run on a different region file
# ============================================================
REGION = "Southern_Asia"          # ← matches the parquet filename stem

# ============================================================
# PATHS
# ============================================================
_DATA = Path("/home/{path}/{fold}/TestFolder/wc_outliers/data_for_outlier")

EMBEDDINGS_DB_PATH = _DATA / "alphaearth.duckdb"
INPUT_PARQUET      = _DATA / f"{REGION}.parquet"

_OUT_DIR = _DATA / f"alphaearth_scores_{REGION}"
_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET     = _DATA / f"{REGION}_with_alphaearth_scores.parquet"
MERGED_SCORES_PATH = _OUT_DIR / f"merged_LC10_CTY24_{REGION}.parquet"

OUT_LC10_DIR  = _OUT_DIR / "LC10"
OUT_CTY24_DIR = _OUT_DIR / "CTY24"
OUT_LC10_DIR.mkdir(parents=True, exist_ok=True)
OUT_CTY24_DIR.mkdir(parents=True, exist_ok=True)

LC10_SAMPLES_PATH  = str(OUT_LC10_DIR / f"LC10_samples_{REGION}.parquet")
LC10_SUMMARY_PATH  = str(OUT_LC10_DIR / f"LC10_summary_{REGION}.parquet")
CTY24_SAMPLES_PATH = str(OUT_CTY24_DIR / f"CTY24_samples_{REGION}.parquet")
CTY24_SUMMARY_PATH = str(OUT_CTY24_DIR / f"CTY24_summary_{REGION}.parquet")

# ============================================================
# ANOMALY COLUMNS (in the same order used throughout the notebook)
# ============================================================
ANOMALY_COLS = [
    "CTY24_confidence_nonoutlier",
    "CTY24_anomaly_flag",
    "outlier_CTY24_cls",
    "LC10_confidence_nonoutlier",
    "LC10_anomaly_flag",
    "outlier_LC10_cls",
]

# ============================================================
# SCORING KNOBS
# ============================================================
# AlphaEarth model has no real hash — use a fixed placeholder
MODEL_HASH_PLACEHOLDER = "alphaearth_v1"

SKIP_CLASSES = ["ignore"]

# LANDCOVER10
LC10_H3_LEVELS    = [2, 3]
LC10_MAX_SLICE    = 10_000
LC10_MIN_SLICE    = 200
LC10_MAX_MERGE    = 16

# CROPTYPE24
CTY24_H3_LEVELS   = [2, 3, 4]
CTY24_MAX_SLICE   = 5_000
CTY24_MIN_SLICE   = 100
CTY24_MAX_MERGE   = 8

# Shared
MAD_K             = 4.0
NORM_PERCENTILES  = (2.0, 98.0)
CENTROID_MODE     = "trimmed"
CENTROID_TRIM     = 0.05

# ============================================================
# CLASS MAPPINGS  — local JSON path (fallback: SharePoint)
# ============================================================
import glob as _glob
_cm_candidates = _glob.glob(
    "/home/wcextractions/.conda/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
) + _glob.glob(
    "/home/{path}/.conda/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
) + _glob.glob(
    "/opt/conda/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
)
LOCAL_CLASS_MAPPINGS_JSON = Path(_cm_candidates[0]) if _cm_candidates else None

SHAREPOINT_ENV_CANDIDATES = [
    Path("/home/wcextractions/.sharepointenv"),
    Path("/home/{path}/{fold}/TestFolder/.sharepointenv"),
]

# ============================================================
# SANITY CHECKS
# ============================================================
assert EMBEDDINGS_DB_PATH.exists(), f"Not found: {EMBEDDINGS_DB_PATH}"
assert INPUT_PARQUET.exists(),      f"Not found: {INPUT_PARQUET}"

print(f"Region            : {REGION}")
print(f"Embeddings DB     : {EMBEDDINGS_DB_PATH}")
print(f"Input parquet     : {INPUT_PARQUET}")
print(f"Output parquet    : {OUTPUT_PARQUET}")
print(f"Merged scores     : {MERGED_SCORES_PATH}")
print(f"LC10 output dir   : {OUT_LC10_DIR}")
print(f"CTY24 output dir  : {OUT_CTY24_DIR}")
print(f"Local class map   : {LOCAL_CLASS_MAPPINGS_JSON}")


## 2) Inspect AlphaEarth Embeddings DuckDB

Verify the table schema, row count, and that `sample_id` and the 64 embedding columns (`A00`–`A63`) are present.

In [ ]:
con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)

print("=== Tables ===")
tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

# Use the first available table (expected: 'raw')
EMBED_TABLE = tables["name"].iloc[0]
print(f"\nUsing table: '{EMBED_TABLE}'")

print("\n=== Schema ===")
schema_df = con.execute(f"DESCRIBE {EMBED_TABLE}").fetchdf()
print(schema_df.to_string())

row_count = con.execute(f"SELECT COUNT(*) FROM {EMBED_TABLE}").fetchone()[0]
print(f"\nTotal rows: {row_count:,}")

# Identify 64 embedding columns (A00 … A63)
all_cols = schema_df["column_name"].tolist()
AE_EMBED_COLS = [c for c in all_cols if len(c) == 3 and c[0] == "A" and c[1:].isdigit()]
AE_EMBED_COLS.sort(key=lambda c: int(c[1:]))
print(f"\nEmbedding columns ({len(AE_EMBED_COLS)}): {AE_EMBED_COLS[:5]} … {AE_EMBED_COLS[-5:]}")
assert len(AE_EMBED_COLS) == 64, f"Expected 64 embedding dims, got {len(AE_EMBED_COLS)}"
assert "sample_id" in all_cols, "'sample_id' column not found!"
assert "lat" in all_cols and "lon" in all_cols, "'lat'/'lon' columns not found!"

print("\n=== Sample rows ===")
sample = con.execute(
    f"SELECT sample_id, lat, lon, {AE_EMBED_COLS[0]}, {AE_EMBED_COLS[1]} "
    f"FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL LIMIT 5"
).fetchdf()
display(sample)

# Count non-null sample_ids
non_null = con.execute(
    f"SELECT COUNT(*) FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL"
).fetchone()[0]
print(f"\nRows with non-null sample_id: {non_null:,} / {row_count:,}")

con.close()


## 2b) Deduplicate DuckDB

The `raw` table may contain multiple rows per `sample_id` (e.g. from repeated GEE exports).  This cell:
1. Reports total vs unique `sample_id` counts and shows the worst offenders.
2. Rewrites the table in-place keeping only **one row per `sample_id`** (NULL rows are dropped).
3. Runs a DuckDB `CHECKPOINT` so the compacted file is persisted to disk.

In [ ]:
# ── Open in READ-ONLY first just to report stats ─────────────────────
con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)

total_rows   = con.execute(f"SELECT COUNT(*) FROM {EMBED_TABLE}").fetchone()[0]
unique_ids   = con.execute(f"SELECT COUNT(DISTINCT sample_id) FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL").fetchone()[0]
null_ids     = con.execute(f"SELECT COUNT(*) FROM {EMBED_TABLE} WHERE sample_id IS NULL").fetchone()[0]
n_duplicates = total_rows - null_ids - unique_ids

print(f"Total rows         : {total_rows:,}")
print(f"Unique sample_id   : {unique_ids:,}")
print(f"NULL sample_id rows: {null_ids:,}")
print(f"Duplicate rows     : {n_duplicates:,}  (to be removed)")

print("\nTop duplicated sample_ids:")
dup_df = con.execute(f"""
    SELECT sample_id, COUNT(*) AS cnt
    FROM {EMBED_TABLE}
    WHERE sample_id IS NOT NULL
    GROUP BY sample_id
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC
    LIMIT 15
""").fetchdf()
print(dup_df.to_string(index=False))
con.close()

# ── Deduplicate in-place if needed ────────────────────────────────────
if n_duplicates > 0 or null_ids > 0:
    print(f"\nDeduplicating {EMBEDDINGS_DB_PATH.name} …")
    con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=False)
    con.execute(f"""
        CREATE OR REPLACE TABLE {EMBED_TABLE} AS
        SELECT * EXCLUDE (_rn)
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY sample_id) AS _rn
            FROM {EMBED_TABLE}
            WHERE sample_id IS NOT NULL
        )
        WHERE _rn = 1
    """)
    new_count = con.execute(f"SELECT COUNT(*) FROM {EMBED_TABLE}").fetchone()[0]
    con.execute("CHECKPOINT")
    con.close()
    print(f"Done. Rows after dedup: {new_count:,}  (removed {total_rows - new_count:,} rows)")
    assert new_count == unique_ids, "Row count mismatch after dedup!"
    print("✓ DuckDB deduplicated and checkpointed.")
else:
    print("\nNo duplicates — DuckDB is already clean.")


## 2c) Regional Coverage — Global Parquet Cross-check

Cross-references the DuckDB `sample_id`s against the full global wide parquet to answer:
- **Which regions** have samples in the DuckDB (and what fraction of each region is covered)?
- **Which `ref_id`s from Southern Asia are missing** — these need to be resubmitted to Google Earth Engine to get AlphaEarth embeddings.

Global parquet: `/projects/worldcereal/data/cached_wide_merged/worldcereal_all_extractions_wide_month_with_anomalies_new.parquet`

In [ ]:
GLOBAL_PARQUET = Path(
    "/projects/worldcereal/data/cached_wide_merged/"
    "worldcereal_all_extractions_wide_month_with_anomalies_new.parquet"
)
assert GLOBAL_PARQUET.exists(), f"Global parquet not found: {GLOBAL_PARQUET}"

# ── Load only the 3 lightweight columns ──────────────────────────────
print(f"Reading sample_id / ref_id / region from global parquet …")
df_global = pd.read_parquet(str(GLOBAL_PARQUET), columns=["sample_id", "ref_id", "region"])
# The wide parquet is already one row per sample, but be safe
df_global.drop_duplicates(subset="sample_id", inplace=True)
df_global.reset_index(drop=True, inplace=True)
print(f"Global parquet unique sample_ids: {len(df_global):,}")
print(f"\nAll regions in global parquet:")
print(df_global["region"].value_counts().to_string())

# ── Load deduplicated DuckDB sample_ids ───────────────────────────────
con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)
db_ids = set(
    con.execute(f"SELECT DISTINCT sample_id FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL")
    .fetchdf()["sample_id"]
    .tolist()
)
con.close()
print(f"\nDuckDB unique sample_ids (after dedup): {len(db_ids):,}")

# ── Coverage by region ────────────────────────────────────────────────
df_global["in_duckdb"] = df_global["sample_id"].isin(db_ids)

coverage = (
    df_global.groupby("region")
    .agg(total=("sample_id", "count"), in_duckdb=("in_duckdb", "sum"))
    .assign(coverage_pct=lambda x: (100 * x["in_duckdb"] / x["total"]).round(1))
    .sort_values("in_duckdb", ascending=False)
)
print("\n=== DuckDB coverage per region ===")
print(coverage[coverage["in_duckdb"] > 0].to_string())

# Warn about regions that should NOT be in DuckDB (i.e., non-Asian regions)
unexpected = coverage[(coverage["in_duckdb"] > 0) & (~coverage.index.str.contains("Asia|Asia", case=False))]
if not unexpected.empty:
    print(f"\n⚠️  Samples from unexpected regions found in DuckDB:")
    print(unexpected.to_string())


In [ ]:
# ── Southern Asia: missing ref_ids → resubmit to GEE ─────────────────
sa_df      = df_global[df_global["region"] == "Southern Asia"].copy()
sa_missing = sa_df[~sa_df["in_duckdb"]].copy()

print(f"=== Southern Asia ===")
print(f"  Total sample_ids : {len(sa_df):,}")
print(f"  In DuckDB        : {int(sa_df['in_duckdb'].sum()):,}  "
      f"({100*sa_df['in_duckdb'].mean():.1f}%)")
print(f"  Missing          : {len(sa_missing):,}  "
      f"({100*len(sa_missing)/len(sa_df):.1f}%)")

print("\nMissing sample counts per ref_id:")
miss_by_ref = (
    sa_missing.groupby("ref_id")
    .size()
    .reset_index(name="missing_samples")
    .sort_values("missing_samples", ascending=False)
)
print(miss_by_ref.to_string(index=False))

# ── Save the missing ref_id list for GEE resubmission ─────────────────
missing_ref_ids_path = _OUT_DIR / f"missing_ref_ids_{REGION}_for_GEE.csv"
miss_by_ref.to_csv(str(missing_ref_ids_path), index=False)
print(f"\nRef_id list saved to: {missing_ref_ids_path}")

# Also save the full list of missing sample_ids (useful for targeted GEE exports)
missing_samples_path = _OUT_DIR / f"missing_sample_ids_{REGION}_for_GEE.csv"
sa_missing[["ref_id", "sample_id"]].to_csv(str(missing_samples_path), index=False)
print(f"Sample_id list saved to: {missing_samples_path}")

# ── Quick sanity check: DuckDB sample_ids that are NOT in the global parquet ─
global_ids = set(df_global["sample_id"])
db_only = db_ids - global_ids
if db_only:
    print(f"\n⚠️  {len(db_only):,} sample_ids in DuckDB are NOT in the global parquet!")
    print("   First 10:", list(db_only)[:10])
else:
    print(f"\n✓ All {len(db_ids):,} DuckDB sample_ids are present in the global parquet.")


## 3) Load & Clean Southern Asia Parquet

Read the parquet, drop existing anomaly columns, and verify sample_id overlap with the embeddings DuckDB.

In [ ]:
df_region = pd.read_parquet(str(INPUT_PARQUET))
print(f"Shape (before cleaning): {df_region.shape}")
print(f"Columns: {df_region.columns.tolist()}")

# ── Drop existing anomaly columns if present ──────────────────────────
cols_to_drop = [c for c in ANOMALY_COLS if c in df_region.columns]
if cols_to_drop:
    df_region.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped existing anomaly columns: {cols_to_drop}")
else:
    print("No existing anomaly columns to drop.")
print(f"Shape (after cleaning) : {df_region.shape}")

# ── Key column checks ────────────────────────────────────────────────
for col in ["ref_id", "sample_id", "ewoc_code", "h3_l3_cell"]:
    if col in df_region.columns:
        print(f"  {col}: {df_region[col].nunique():,} unique values")
    else:
        print(f"  WARNING: '{col}' NOT in parquet!")

print(f"\nUnique sample_ids in parquet : {df_region['sample_id'].nunique():,}")
print(f"\newoc_code value counts (top 15):")
print(df_region["ewoc_code"].value_counts().head(15))

# ── Overlap with DuckDB ──────────────────────────────────────────────
con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)
db_ids = set(
    con.execute(f"SELECT DISTINCT sample_id FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL")
    .fetchdf()["sample_id"]
    .tolist()
)
con.close()

parquet_ids = set(df_region["sample_id"].dropna().unique())
overlap = parquet_ids & db_ids
print(f"\nParquet sample_ids   : {len(parquet_ids):,}")
print(f"DuckDB sample_ids    : {len(db_ids):,}")
print(f"Overlap              : {len(overlap):,} ({100*len(overlap)/max(len(parquet_ids),1):.1f}% of parquet)")


## 4) Prepare Embeddings DataFrame

Load the AlphaEarth embeddings, rename `A00`–`A63` → `embedding_0`–`embedding_63`, add a synthetic `model_hash`, and join with parquet metadata (`ref_id`, `ewoc_code`, `h3_l3_cell`) on `sample_id`.

If `h3_l3_cell` is absent from the DuckDB table it is computed from `lat`/`lon` using the `h3` library at resolution 3.

In [ ]:
# ── Load all embeddings from DuckDB ──────────────────────────────────
print(f"Loading embeddings from {EMBEDDINGS_DB_PATH} …")
con = duckdb.connect(str(EMBEDDINGS_DB_PATH), read_only=True)
select_cols = ["sample_id", "lat", "lon"] + AE_EMBED_COLS
# Only load rows with non-null sample_id
df_emb = con.execute(
    f"SELECT {', '.join(select_cols)} FROM {EMBED_TABLE} WHERE sample_id IS NOT NULL"
).fetchdf()
con.close()
print(f"Loaded {len(df_emb):,} rows from DuckDB")

# ── Rename A00…A63 → embedding_0…embedding_63 ────────────────────────
rename_map = {ae: f"embedding_{i}" for i, ae in enumerate(AE_EMBED_COLS)}
df_emb.rename(columns=rename_map, inplace=True)
embed_cols = [f"embedding_{i}" for i in range(len(AE_EMBED_COLS))]
print(f"Renamed {len(embed_cols)} embedding columns: {embed_cols[:3]} … {embed_cols[-3:]}")

# ── Cast embeddings to float32 ────────────────────────────────────────
df_emb[embed_cols] = df_emb[embed_cols].astype("float32")

# ── Add synthetic model_hash ──────────────────────────────────────────
df_emb["model_hash"] = MODEL_HASH_PLACEHOLDER

# ── Build parquet metadata lookup: sample_id → ref_id, ewoc_code, h3_l3_cell ──
meta_cols = ["sample_id", "ref_id", "ewoc_code"]
if "h3_l3_cell" in df_region.columns:
    meta_cols.append("h3_l3_cell")
if "lat" in df_region.columns and "lon" in df_region.columns:
    meta_cols += ["lat", "lon"]

df_meta = (
    df_region[meta_cols]
    .drop_duplicates(subset="sample_id")
    .copy()
)
print(f"Metadata rows (unique sample_id): {len(df_meta):,}")

# ── Left-join embeddings with metadata ───────────────────────────────
# Use parquet lat/lon if DuckDB lat/lon is missing; prefer DuckDB values otherwise
df_emb = df_emb.merge(df_meta, on="sample_id", how="left", suffixes=("", "_meta"))

# Consolidate lat/lon: prefer DuckDB values, fall back to parquet
for coord in ("lat", "lon"):
    meta_col = f"{coord}_meta"
    if meta_col in df_emb.columns:
        df_emb[coord] = df_emb[coord].combine_first(df_emb[meta_col])
        df_emb.drop(columns=[meta_col], inplace=True)

print(f"After join: {len(df_emb):,} rows")
null_ref = df_emb["ref_id"].isna().sum()
null_ewoc = df_emb["ewoc_code"].isna().sum()
print(f"  Null ref_id   : {null_ref:,}  (no parquet metadata match)")
print(f"  Null ewoc_code: {null_ewoc:,}")

# ── Compute h3_l3_cell if not already present ─────────────────────────
if "h3_l3_cell" not in df_emb.columns or df_emb["h3_l3_cell"].isna().all():
    print("Computing h3_l3_cell from lat/lon at resolution 3 …")
    try:
        import h3
        df_emb["h3_l3_cell"] = df_emb.apply(
            lambda r: h3.latlng_to_cell(r["lat"], r["lon"], 3)
            if pd.notna(r["lat"]) and pd.notna(r["lon"]) else None,
            axis=1,
        )
        print(f"  h3_l3_cell computed — {df_emb['h3_l3_cell'].notna().sum():,} non-null")
    except ImportError:
        print("  WARNING: h3 library not installed — h3_l3_cell will remain null.")
        df_emb["h3_l3_cell"] = None
else:
    print(f"h3_l3_cell present — {df_emb['h3_l3_cell'].notna().sum():,} non-null")

# ── Ensure ewoc_code is a string (as expected by run_pipeline) ─────────
df_emb["ewoc_code"] = df_emb["ewoc_code"].astype(str)

# ── Drop rows with no parquet metadata match ──────────────────────────
df_emb = df_emb[df_emb["ref_id"].notna()].copy()
print(f"\nFinal embeddings DataFrame shape: {df_emb.shape}")
print(f"Columns: {df_emb.columns.tolist()}")
display(df_emb[["sample_id", "ref_id", "ewoc_code", "h3_l3_cell", "lat", "lon",
                 embed_cols[0], embed_cols[1]]].head())


## 5) Load Class Mappings

Tries a local `class_mappings.json` first (found automatically in the active conda env's `worldcereal` package).  Falls back to fetching from SharePoint if the local file is not found.

In [ ]:
if LOCAL_CLASS_MAPPINGS_JSON is not None and LOCAL_CLASS_MAPPINGS_JSON.exists():
    print(f"Loading class mappings from local file:\n  {LOCAL_CLASS_MAPPINGS_JSON}")
    with open(LOCAL_CLASS_MAPPINGS_JSON) as _f:
        CLASS_MAPPINGS = json.load(_f)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))

else:
    print("Local class_mappings.json not found — fetching from SharePoint …")
    from dotenv import load_dotenv
    from worldcereal.utils.sharepoint import get_excel_from_sharepoint, build_class_mappings

    _env_path = next((p for p in SHAREPOINT_ENV_CANDIDATES if p.exists()), None)
    assert _env_path is not None, (
        f".sharepointenv not found at: {[str(p) for p in SHAREPOINT_ENV_CANDIDATES]}"
    )
    print(f"Using .sharepointenv: {_env_path}")
    load_dotenv(_env_path, override=True)

    legend = get_excel_from_sharepoint(
        site_url=os.environ["WORLDCEREAL_SP_SITE_URL"],
        file_server_relative_url=os.environ["WORLDCEREAL_SP_FILE_URL"],
        retries=10,
        sheet_name=0,
    )
    legend["ewoc_code"] = (
        legend["ewoc_code"]
        .astype("string")
        .str.replace("-", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )
    CLASS_MAPPINGS = build_class_mappings(legend)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))

assert "LANDCOVER10" in CLASS_MAPPINGS, "LANDCOVER10 key missing from CLASS_MAPPINGS!"
assert "CROPTYPE24"  in CLASS_MAPPINGS, "CROPTYPE24 key missing from CLASS_MAPPINGS!"
print("\nClass mappings loaded successfully.")


## 6) Scoring — LANDCOVER10

Runs the outlier pipeline on AlphaEarth embeddings for the `LANDCOVER10` label schema by passing `embeddings_df=(df_emb, embed_cols)`.  This bypasses the DuckDB reads inside `run_pipeline` entirely.

In [ ]:
LC10_flagged_gdf, LC10_summary_df = run_pipeline(
    embeddings_db_path=None,          # not used — embeddings_df takes precedence
    restrict_model_hash=None,
    label_domain="LANDCOVER10",
    map_to_finetune=False,
    class_mappings_name="LANDCOVER10",
    skip_classes=SKIP_CLASSES,
    mapping_file=CLASS_MAPPINGS,
    h3_level=LC10_H3_LEVELS,
    group_cols=None,
    min_slice_size=LC10_MIN_SLICE,
    max_slice_size=LC10_MAX_SLICE,
    merge_small_slice=True,
    max_merge_iterations=LC10_MAX_MERGE,
    threshold_mode="mad",
    percentile_q=0.96,
    mad_k=MAD_K,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,
    norm_percentiles=NORM_PERCENTILES,
    output_samples_path=LC10_SAMPLES_PATH,
    output_summary_path=LC10_SUMMARY_PATH,
    debug=False,
    centroid_mode=CENTROID_MODE,
    centroid_trim=CENTROID_TRIM,
    gate_confidence_by_flag=True,
    apply_slice_trust=False,
    slice_trust_min=0.05,
    embeddings_df=(df_emb, embed_cols),   # ← inject AlphaEarth embeddings directly
)
print(f"LC10 pipeline done — {len(LC10_flagged_gdf):,} samples scored.")


In [ ]:
# ── Read back & rename for downstream merging ─────────────────────────
# (also useful for resuming the notebook without re-running run_pipeline)
_lc10_cols = ["ref_id", "sample_id", "LANDCOVER10", "confidence_nonoutlier", "anomaly_flag"]
LC10_flagged_gdf = pd.read_parquet(LC10_SAMPLES_PATH, columns=_lc10_cols)

LC10_flagged_gdf = LC10_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "LC10_confidence_nonoutlier",
    "anomaly_flag":          "LC10_anomaly_flag",
    "LANDCOVER10":           "outlier_LC10_cls",
})
print(f"LC10 scores: {len(LC10_flagged_gdf):,} rows")
print(LC10_flagged_gdf["LC10_anomaly_flag"].value_counts())
display(LC10_flagged_gdf.head())


## 7) Scoring — CROPTYPE24

Same approach with the finer `CROPTYPE24` schema: three H3 levels and a smaller per-slice cap.

In [ ]:
CTY24_flagged_gdf, CTY24_summary_df = run_pipeline(
    embeddings_db_path=None,          # not used — embeddings_df takes precedence
    restrict_model_hash=None,
    label_domain="CROPTYPE24",
    map_to_finetune=False,
    class_mappings_name="CROPTYPE24",
    skip_classes=SKIP_CLASSES,
    mapping_file=CLASS_MAPPINGS,
    h3_level=CTY24_H3_LEVELS,
    group_cols=None,
    min_slice_size=CTY24_MIN_SLICE,
    max_slice_size=CTY24_MAX_SLICE,
    merge_small_slice=True,
    max_merge_iterations=CTY24_MAX_MERGE,
    threshold_mode="mad",
    percentile_q=0.96,
    mad_k=MAD_K,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,
    norm_percentiles=NORM_PERCENTILES,
    output_samples_path=CTY24_SAMPLES_PATH,
    output_summary_path=CTY24_SUMMARY_PATH,
    debug=False,
    centroid_mode=CENTROID_MODE,
    centroid_trim=CENTROID_TRIM,
    gate_confidence_by_flag=True,
    apply_slice_trust=False,
    slice_trust_min=0.05,
    embeddings_df=(df_emb, embed_cols),   # ← inject AlphaEarth embeddings directly
)
print(f"CTY24 pipeline done — {len(CTY24_flagged_gdf):,} samples scored.")


In [ ]:
# ── Read back & rename for downstream merging ─────────────────────────
_cty24_cols = ["ref_id", "sample_id", "CROPTYPE24", "confidence_nonoutlier", "anomaly_flag"]
CTY24_flagged_gdf = pd.read_parquet(CTY24_SAMPLES_PATH, columns=_cty24_cols)

CTY24_flagged_gdf = CTY24_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "CTY24_confidence_nonoutlier",
    "anomaly_flag":          "CTY24_anomaly_flag",
    "CROPTYPE24":            "outlier_CTY24_cls",
})
print(f"CTY24 scores: {len(CTY24_flagged_gdf):,} rows")
print(CTY24_flagged_gdf["CTY24_anomaly_flag"].value_counts())
display(CTY24_flagged_gdf.head())


## 8) Merge LC10 + CTY24 Scores

Outer-join the two score tables on `(ref_id, sample_id)`.  Handles the case where one DataFrame is empty.

In [ ]:
if LC10_flagged_gdf.empty and CTY24_flagged_gdf.empty:
    print("Both score DataFrames are empty — nothing to merge.")
    merged_scores = pd.DataFrame(columns=["ref_id", "sample_id"] + ANOMALY_COLS)
elif LC10_flagged_gdf.empty:
    merged_scores = CTY24_flagged_gdf.copy()
    for col in ["LC10_confidence_nonoutlier", "LC10_anomaly_flag", "outlier_LC10_cls"]:
        merged_scores[col] = float("nan") if "confidence" in col else None
elif CTY24_flagged_gdf.empty:
    merged_scores = LC10_flagged_gdf.copy()
    for col in ["CTY24_confidence_nonoutlier", "CTY24_anomaly_flag", "outlier_CTY24_cls"]:
        merged_scores[col] = float("nan") if "confidence" in col else None
else:
    merged_scores = CTY24_flagged_gdf.merge(
        LC10_flagged_gdf, on=["ref_id", "sample_id"], how="outer"
    )

merged_scores.sort_values(["ref_id", "sample_id"], inplace=True)
merged_scores.reset_index(drop=True, inplace=True)

print(f"Merged scores: {len(merged_scores):,} rows × {merged_scores.shape[1]} columns")
print("\nLC10_anomaly_flag:")
print(merged_scores["LC10_anomaly_flag"].value_counts(dropna=False))
print("\nCTY24_anomaly_flag:")
print(merged_scores["CTY24_anomaly_flag"].value_counts(dropna=False))

# ── Save to disk ───────────────────────────────────────────────────────
merged_scores.to_parquet(str(MERGED_SCORES_PATH), index=False)
print(f"\nMerged scores saved to: {MERGED_SCORES_PATH}")

display(merged_scores.head())


## 8b) Post-processing: skip-class fill + LC10→CTY24 escalation

Two adjustments applied to `merged_scores` **after** scoring, before writing to disk:

### Skip-class fill
Samples mapped to a skip class (e.g. `"ignore"`) are held aside by `run_pipeline` and return `NaN` in all anomaly columns.  They are definitively **not** outliers, so their scores are filled with `confidence_nonoutlier = 1.0` and `anomaly_flag = "normal"` for both domains.

### LC10 → CTY24 escalation voting
When a sample's LC10 label is `temporary_crops` **and** its LC10 anomaly flag is strictly higher than its CTY24 flag **and** CTY24 is not already `"normal"`, the CTY24 flag is raised by one level (capped at the LC10 level).  Confidence is averaged between the two domains for escalated rows.  This flows strictly from the parent class (`temporary_crops`) to its sub-crop categories, never in the reverse direction, and never promotes a `normal` CTY24 result.


In [ ]:

# ============================================================
# Post-processing helpers (shared by both Presto and AlphaEarth workflows)
# ============================================================

_FLAG_ORDER = ["normal", "flagged", "suspect", "candidate"]
_FLAG_RANK  = {f: i for i, f in enumerate(_FLAG_ORDER)}

# ── Post-processing skip classes ──────────────────────────────────────
# Any sample whose LC10 class OR CTY24 class appears in this list will
# have its scores reset to confidence=1.0 / flag="normal" for the
# matching domain AFTER scoring.  Independent of run_pipeline skip_classes.
# Examples: ["built-up", "water", "bare_soil", "ignore"]
POST_PROCESSING_SKIP_CLASSES: list[str] = []   # ← edit as needed


def _fill_skip_class_scores(df: pd.DataFrame) -> pd.DataFrame:
    """Fill NaN anomaly values (skip-class rows) with confidence=1.0 / flag='normal'."""
    df = df.copy()
    for prefix in ("LC10", "CTY24"):
        conf_col = f"{prefix}_confidence_nonoutlier"
        flag_col = f"{prefix}_anomaly_flag"
        if conf_col in df.columns:
            df[conf_col] = df[conf_col].fillna(1.0).astype("float32")
        if flag_col in df.columns:
            df[flag_col] = df[flag_col].fillna("normal")
    return df


def _apply_postprocessing_skip_classes(
    df: pd.DataFrame,
    skip_classes: list[str],
) -> pd.DataFrame:
    """Reset scores to 1.0 / 'normal' for samples whose scored class is in skip_classes.

    Applied after scoring — for each domain independently:
    - If ``outlier_LC10_cls`` is in skip_classes → LC10 confidence=1.0, flag='normal'
    - If ``outlier_CTY24_cls`` is in skip_classes → CTY24 confidence=1.0, flag='normal'
    """
    if not skip_classes:
        return df

    df = df.copy()
    skip_set = {str(c).lower().strip() for c in skip_classes}

    domain_map = {
        "LC10":  ("outlier_LC10_cls",  "LC10_confidence_nonoutlier",  "LC10_anomaly_flag"),
        "CTY24": ("outlier_CTY24_cls", "CTY24_confidence_nonoutlier", "CTY24_anomaly_flag"),
    }

    for domain, (cls_col, conf_col, flag_col) in domain_map.items():
        if cls_col not in df.columns:
            continue
        mask = df[cls_col].astype(str).str.lower().str.strip().isin(skip_set)
        n = int(mask.sum())
        if n == 0:
            continue
        if conf_col in df.columns:
            df.loc[mask, conf_col] = float(1.0)
            df[conf_col] = df[conf_col].astype("float32")
        if flag_col in df.columns:
            df.loc[mask, flag_col] = "normal"
        print(f"[post-skip/{domain}] Reset {n:,} rows matching classes: "
              f"{df.loc[mask, cls_col].value_counts().to_dict()}")

    return df


def _apply_lc10_to_cty24_escalation(df: pd.DataFrame) -> pd.DataFrame:
    """Escalate CTY24 anomaly flag when LC10=temporary_crops is scored higher."""
    df = df.copy()

    lc10_cls_col   = "outlier_LC10_cls"
    lc10_flag_col  = "LC10_anomaly_flag"
    lc10_conf_col  = "LC10_confidence_nonoutlier"
    cty24_flag_col = "CTY24_anomaly_flag"
    cty24_conf_col = "CTY24_confidence_nonoutlier"

    required = [lc10_cls_col, lc10_flag_col, lc10_conf_col, cty24_flag_col, cty24_conf_col]
    if not all(c in df.columns for c in required):
        print("[escalation] Missing required columns — skipping LC10→CTY24 escalation.")
        return df

    lc10_rank  = df[lc10_flag_col].map(_FLAG_RANK).fillna(0).astype(int)
    cty24_rank = df[cty24_flag_col].map(_FLAG_RANK).fillna(0).astype(int)

    esc_mask = (
        (df[lc10_cls_col].astype(str).str.lower() == "temporary_crops")
        & (cty24_rank > 0)
        & (lc10_rank > cty24_rank)
    )

    n_esc = int(esc_mask.sum())
    print(f"[escalation] {n_esc:,} samples eligible for LC10→CTY24 escalation.")

    if n_esc == 0:
        return df

    new_cty24_rank = (cty24_rank + 1).clip(upper=lc10_rank)
    df.loc[esc_mask, cty24_flag_col] = new_cty24_rank[esc_mask].map(
        lambda r: _FLAG_ORDER[int(r)]
    )

    avg_conf = (
        df.loc[esc_mask, lc10_conf_col].astype(float)
        + df.loc[esc_mask, cty24_conf_col].astype(float)
    ) / 2.0
    df.loc[esc_mask, cty24_conf_col] = avg_conf.clip(0.0, 1.0).astype("float32")

    print(
        "[escalation] Escalated CTY24 flag distribution:\n"
        + df.loc[esc_mask, cty24_flag_col].value_counts().to_string()
    )
    return df


# ── Apply post-processing to merged_scores ──────────────────────────
if not merged_scores.empty:
    merged_scores = _fill_skip_class_scores(merged_scores)
    if POST_PROCESSING_SKIP_CLASSES:
        merged_scores = _apply_postprocessing_skip_classes(merged_scores, POST_PROCESSING_SKIP_CLASSES)
    merged_scores = _apply_lc10_to_cty24_escalation(merged_scores)
    print(f"\nPost-processing complete — {len(merged_scores):,} rows.")
    print("\nLC10_anomaly_flag after post-processing:")
    print(merged_scores["LC10_anomaly_flag"].value_counts(dropna=False))
    print("\nCTY24_anomaly_flag after post-processing:")
    print(merged_scores["CTY24_anomaly_flag"].value_counts(dropna=False))
    # Overwrite on disk so section 9 picks up the post-processed scores
    merged_scores.sort_values(["ref_id", "sample_id"], inplace=True)
    merged_scores.reset_index(drop=True, inplace=True)
    merged_scores.to_parquet(str(MERGED_SCORES_PATH), index=False)
    print(f"Post-processed scores written to: {MERGED_SCORES_PATH}")
else:
    print("merged_scores is empty — post-processing skipped.")


## 8c) Export merged scores as GeoParquet (QGIS-ready)

Attaches coordinates to `merged_scores` (from the embeddings DuckDB or the input parquet) and writes a GeoParquet file that can be opened directly in **QGIS**, shared with collaborators, or loaded with `geopandas.read_parquet()`.


In [ ]:

import geopandas as gpd
from shapely.geometry import Point

# ── Output path ────────────────────────────────────────────────────────
GEOPARQUET_SCORES_PATH = MERGED_SCORES_PATH.parent / (MERGED_SCORES_PATH.stem + "_geo.geoparquet")
GEOPARQUET_CRS = "EPSG:4326"   # WGS84 — change to reproject (e.g. "EPSG:3857")

# ── Attach coordinates to merged_scores ───────────────────────────────
# AlphaEarth: lat/lon are available in df_emb (loaded from DuckDB).
# We join from there; if df_emb is not in scope, fall back to df_region.
_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

for _latlon_src_name, _latlon_src in [("df_emb", locals().get("df_emb")),
                                       ("df_region", locals().get("df_region"))]:
    if _latlon_src is not None and "lat" in _latlon_src.columns and "lon" in _latlon_src.columns:
        _latlon = (
            _latlon_src[["sample_id", "lat", "lon"]]
            .drop_duplicates(subset="sample_id")
            .copy()
        )
        _scores = _scores.merge(_latlon, on="sample_id", how="left")
        print(f"Coordinates joined from '{_latlon_src_name}'")
        break
else:
    print("WARNING: could not find lat/lon source — run sections 3–4 first.")

# ── Build & write GeoDataFrame ────────────────────────────────────────
if "lat" in _scores.columns and "lon" in _scores.columns:
    _df_geo = _scores.dropna(subset=["lat", "lon"]).copy()
    n_missing = len(_scores) - len(_df_geo)
    if n_missing:
        print(f"  Dropped {n_missing:,} rows with null lat/lon.")

    gdf_scores = gpd.GeoDataFrame(
        _df_geo,
        geometry=[Point(lon, lat) for lon, lat in zip(_df_geo["lon"], _df_geo["lat"])],
        crs=GEOPARQUET_CRS,
    )
    gdf_scores.to_parquet(str(GEOPARQUET_SCORES_PATH))
    print(f"GeoParquet written : {GEOPARQUET_SCORES_PATH}")
    print(f"  Rows    : {len(gdf_scores):,}")
    print(f"  CRS     : {gdf_scores.crs}")
    print(f"\n  LC10_anomaly_flag:")
    print(gdf_scores["LC10_anomaly_flag"].value_counts(dropna=False).to_string())
    print(f"\n  CTY24_anomaly_flag:")
    print(gdf_scores["CTY24_anomaly_flag"].value_counts(dropna=False).to_string())
    print(f"\n→ Open in QGIS: drag-and-drop {GEOPARQUET_SCORES_PATH.name} into the QGIS window.")
else:
    print("Skipping GeoParquet export — no lat/lon available.")


## 9) Write Scores Back to Parquet

Left-join the merged scores onto the cleaned `Southern_Asia` DataFrame and write to `OUTPUT_PARQUET`.

In [ ]:
# ── Read merged scores from disk (safe even if above cells were skipped) ──
merged_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

# ── Build lookup: one row per sample_id ──────────────────────────────
scores_lookup = (
    merged_scores[["ref_id", "sample_id"] + ANOMALY_COLS]
    .drop_duplicates(subset="sample_id")
    .copy()
)
print(f"Scores lookup: {len(scores_lookup):,} unique sample_ids")

# ── Ensure old anomaly columns are gone from df_region ───────────────
cols_to_drop = [c for c in ANOMALY_COLS if c in df_region.columns]
if cols_to_drop:
    df_region.drop(columns=cols_to_drop, inplace=True)

# ── Left-join scores onto the parquet rows ────────────────────────────
df_out = df_region.merge(scores_lookup, on=["ref_id", "sample_id"], how="left")
print(f"Output DataFrame shape: {df_out.shape}")

# ── Join completeness check ───────────────────────────────────────────
for col in ANOMALY_COLS:
    n_null = df_out[col].isna().sum()
    pct = 100 * n_null / max(len(df_out), 1)
    print(f"  {col}: {n_null:,} NaN ({pct:.1f}%)")

# ── Write output ──────────────────────────────────────────────────────
df_out.to_parquet(
    str(OUTPUT_PARQUET),
    index=False,
    compression="zstd",
    row_group_size=20_000,
)
print(f"\nOutput written to: {OUTPUT_PARQUET}")
print(f"Final shape: {df_out.shape}")

print("\n=== LC10_anomaly_flag distribution ===")
print(df_out["LC10_anomaly_flag"].value_counts(dropna=False))
print("\n=== CTY24_anomaly_flag distribution ===")
print(df_out["CTY24_anomaly_flag"].value_counts(dropna=False))
